In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

plt.rcParams["figure.figsize"] = (8, 4)

TARGET = "IsBadBuy"
PROCESSED_PATH = "../data/processed/s1_base.csv"


In [ ]:
def basic_cleaning_temp(df):
    """
    LIMPIEZA TEMPORAL.
    Reemplazar luego por el output de 02.
    """
    df = df.copy()

    df.columns = [c.strip() for c in df.columns]

    obj_cols = df.select_dtypes(include="object").columns
    for col in obj_cols:
        df[col] = df[col].astype(str).str.strip()
        df[col] = df[col].replace({"nan": np.nan, "None": np.nan, "": np.nan})

    if "PurchDate" in df.columns:
        df["PurchDate"] = pd.to_datetime(df["PurchDate"], errors="coerce")

    if TARGET in df.columns:
        df[TARGET] = pd.to_numeric(df[TARGET], errors="coerce")
        df = df[df[TARGET].notna()]
        df[TARGET] = df[TARGET].astype(int)

    cat_cols = df.select_dtypes(include="object").columns
    for col in cat_cols:
        df[col] = df[col].fillna("Missing")

    num_cols = df.select_dtypes(include=[np.number]).columns
    for col in num_cols:
        if col != TARGET:
            df[col] = df[col].fillna(df[col].median())

    return df

In [ ]:
DATA_PATH = "../data/raw/06-kickAutomotriz.csv"

df = pd.read_csv(DATA_PATH)

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df.select_dtypes(include="object").columns.tolist()

if TARGET in numeric_cols:
    numeric_cols.remove(TARGET)

print("Shape:", df.shape)
display(df.head())

Shape: (4, 1)


,outs:
0,- md5: 7c38ca8034f74c2e517613abf95c7fcf
1,size: 13547043
2,hash: md5
3,path: 06-kickAutomotriz.csv


In [ ]:
plot_type = widgets.Dropdown(
    options=["Histogram", "Bar", "Target Rate", "Summary Table"],
    value="Histogram",
    description="Vista:"
)

variable_selector = widgets.Dropdown(
    options=sorted(df.columns.tolist()),
    value=sorted(df.columns.tolist())[0],
    description="Variable:"
)

target_filter = widgets.Dropdown(
    options=["Todos", "Solo IsBadBuy=0", "Solo IsBadBuy=1"],
    value="Todos",
    description="Filtro:"
)

top_n = widgets.IntSlider(
    value=10,
    min=5,
    max=20,
    step=1,
    description="Top N:"
)

output = widgets.Output()

In [ ]:
def apply_target_filter(df_in, target_filter_value):
    if target_filter_value == "Solo IsBadBuy=0":
        return df_in[df_in[TARGET] == 0]
    elif target_filter_value == "Solo IsBadBuy=1":
        return df_in[df_in[TARGET] == 1]
    return df_in


def render_prototype(plot_type_value, variable_value, target_filter_value, top_n_value):
    with output:
        clear_output(wait=True)

        temp = apply_target_filter(df, target_filter_value)

        print(f"Filas analizadas: {len(temp)}")
        print(f"Variable seleccionada: {variable_value}")

        if plot_type_value == "Summary Table":
            display(temp[[variable_value]].describe(include="all").T)
            return

        if variable_value in numeric_cols:
            if plot_type_value == "Histogram":
                plt.figure()
                plt.hist(temp[variable_value].dropna(), bins=30)
                plt.title(f"Distribución de {variable_value}")
                plt.xlabel(variable_value)
                plt.ylabel("Frecuencia")
                plt.show()

            elif plot_type_value == "Bar":
                bins = pd.cut(temp[variable_value], bins=10)
                counts = bins.value_counts().sort_index()
                plt.figure()
                plt.bar(counts.index.astype(str), counts.values)
                plt.title(f"{variable_value} agrupada en bins")
                plt.xlabel(variable_value)
                plt.ylabel("Frecuencia")
                plt.xticks(rotation=45, ha="right")
                plt.show()

            elif plot_type_value == "Target Rate":
                if TARGET in df.columns:
                    bins = pd.cut(temp[variable_value], bins=10)
                    rate = temp.groupby(bins)[TARGET].mean()
                    plt.figure()
                    plt.bar(rate.index.astype(str), rate.values)
                    plt.title(f"Tasa promedio de {TARGET}=1 por bins de {variable_value}")
                    plt.xlabel(variable_value)
                    plt.ylabel("Target rate")
                    plt.xticks(rotation=45, ha="right")
                    plt.show()

        else:
            counts = temp[variable_value].value_counts(dropna=False).head(top_n_value)

            if plot_type_value in ["Bar", "Histogram"]:
                plt.figure()
                plt.bar(counts.index.astype(str), counts.values)
                plt.title(f"Top categorías de {variable_value}")
                plt.xlabel(variable_value)
                plt.ylabel("Frecuencia")
                plt.xticks(rotation=45, ha="right")
                plt.show()

            elif plot_type_value == "Target Rate":
                if TARGET in df.columns:
                    rate = (
                        temp.groupby(variable_value)[TARGET]
                        .mean()
                        .sort_values(ascending=False)
                        .head(top_n_value)
                    )
                    plt.figure()
                    plt.bar(rate.index.astype(str), rate.values)
                    plt.title(f"Tasa promedio de {TARGET}=1 por {variable_value}")
                    plt.xlabel(variable_value)
                    plt.ylabel("Target rate")
                    plt.xticks(rotation=45, ha="right")
                    plt.show()

In [ ]:
controls = widgets.VBox([
    plot_type,
    variable_selector,
    target_filter,
    top_n
])

ui = widgets.HBox([controls, output])

def on_change(change=None):
    render_prototype(
        plot_type.value,
        variable_selector.value,
        target_filter.value,
        top_n.value
    )

plot_type.observe(on_change, names="value")
variable_selector.observe(on_change, names="value")
target_filter.observe(on_change, names="value")
top_n.observe(on_change, names="value")

display(ui)
on_change()